# Stability Consensus Analysis on EMOS CIF Dataset

This notebook runs `StabilityConsensusAnalysisFeature` on CIF files from the EMOS stability dataset and exports analysis outputs (JSON and CSV) into this analysis folder.

In [1]:
import csv
import json
import os
import sys
import time
from pathlib import Path

workspace_root = Path(os.path.abspath("")).resolve()
while workspace_root != workspace_root.parent:
    if (workspace_root / "Features").is_dir() and (workspace_root / "Information_Units").is_dir():
        break
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from Features.Materials_Exploration.StabilityConsensusAnalysis.StabilityConsensusAnalysisFeature import StabilityConsensusAnalysisFeature

analysis_root = workspace_root / "Features" / "Materials_Exploration" / "StabilityConsensusAnalysis" / "analysis"
analysis_root.mkdir(parents=True, exist_ok=True)

print(f"Workspace root: {workspace_root}")
print(f"Analysis output folder: {analysis_root}")

ModuleNotFoundError: No module named 'pymatgen'

In [ ]:
dataset_candidates = [
    workspace_root / "Features" / "Materials_Exploration" / "StabilityConsensusAnalysis" / "emos_cif_files" / "emos_stability_dataset",
    workspace_root / "Features" / "Materials_Exploration" / "StabilityConsensusAnalysis" / "emos_cif_files" / "emos_cif_files" / "emos_stability_dataset",
]

dataset_dir = next((path for path in dataset_candidates if path.is_dir()), None)
if dataset_dir is None:
    raise FileNotFoundError(
        "Could not find dataset directory. Checked: " + 
.join(str(p) for p in dataset_candidates)
    )

cif_paths = sorted(dataset_dir.glob("*.cif"))
if not cif_paths:
    raise FileNotFoundError(f"No CIF files found in: {dataset_dir}")

print(f"Dataset folder: {dataset_dir}")
print(f"Number of CIF files found: {len(cif_paths)}")
print("Sample files:")
for p in cif_paths[:10]:
    print(f"  - {p.name}")

In [ ]:
def load_cif_entries(paths, max_files=None):
    selected_paths = paths if max_files is None else paths[:max_files]
    entries = []
    for path in selected_paths:
        entries.append({
            "name": path.name,
            "content": path.read_text(encoding="utf-8", errors="ignore"),
        })
    return entries

MAX_FILES = None
cif_entries = load_cif_entries(cif_paths, max_files=MAX_FILES)
print(f"Prepared CIF entries for analysis: {len(cif_entries)}")

In [ ]:
ACTIVE_DATABASES = [
    {"value": "materialsproject", "name": "Materials Project"},
    {"value": "alexandria", "name": "Alexandria"},
]

# Keep predictor calls optional in case runtime dependencies/services are unavailable.
USE_PREDICTORS = False
ACTIVE_PREDICTORS = [
    {"value": "mattersim", "name": "MatterSim"},
    {"value": "chgnet", "name": "CHGNet"},
] if USE_PREDICTORS else []

payload = {
    "cif_files": cif_entries,
    "active_databases": ACTIVE_DATABASES,
    "active_predictors": ACTIVE_PREDICTORS,
}

print("Configured payload:")
print(f"  CIF files: {len(payload['cif_files'])}")
print(f"  Databases: {[d['value'] for d in payload['active_databases']]}")
print(f"  Predictors: {[p['value'] for p in payload['active_predictors']]}")

In [ ]:
feature = StabilityConsensusAnalysisFeature()
inputs = feature.extract_inputs(payload)

started = time.time()
raw_results = feature.process_feature(inputs)
elapsed_seconds = round(time.time() - started, 2)

formatted_results = feature.format_outputs(raw_results)
results_per_cif = formatted_results.get("results_per_cif", [])
batch_summary = formatted_results.get("batch_summary", {})
plot_data = formatted_results.get("plot_data", {})

print(f"Feature runtime (s): {elapsed_seconds}")
print("Batch summary:")
print(json.dumps(batch_summary, indent=2))
print(f"Results entries returned: {len(results_per_cif)}")

In [ ]:
def parse_stability_label(label):
    if not isinstance(label, str):
        return "unknown"
    lowered = label.lower()
    if "unstable" in lowered:
        return "unstable"
    if "stable" in lowered:
        return "stable"
    if "not found" in lowered:
        return "not_found"
    return "other"

summary_rows = []
source_rows = []

for result in results_per_cif:
    cif_name = result.get("cif_name", "")
    error_text = result.get("error", "")
    summary = result.get("summary", {}) or {}

    summary_rows.append({
        "cif_name": cif_name,
        "composition": result.get("composition", ""),
        "space_group_number": result.get("space_group_number", ""),
        "space_group_symbol": result.get("space_group_symbol", ""),
        "materialsproject_id": result.get("materialsproject_id", ""),
        "stable_count": summary.get("stable_count", 0),
        "unstable_count": summary.get("unstable_count", 0),
        "total_sources": summary.get("total_sources", 0),
        "consensus": summary.get("consensus", ""),
        "error": error_text,
    })

    for source_name, source_data in (result.get("sources", {}) or {}).items():
        label = source_data.get("stability") if isinstance(source_data, dict) else None
        source_rows.append({
            "cif_name": cif_name,
            "composition": result.get("composition", ""),
            "source": source_name,
            "status": source_data.get("status") if isinstance(source_data, dict) else None,
            "stability": label,
            "stability_class": parse_stability_label(label),
            "raw_value": source_data.get("raw_value") if isinstance(source_data, dict) else None,
            "threshold": source_data.get("threshold") if isinstance(source_data, dict) else None,
            "unit": source_data.get("unit") if isinstance(source_data, dict) else None,
            "description": source_data.get("description") if isinstance(source_data, dict) else None,
            "message": source_data.get("message") if isinstance(source_data, dict) else None,
        })

print(f"Summary rows: {len(summary_rows)}")
print(f"Per-source rows: {len(source_rows)}")

In [ ]:
json_path = analysis_root / "stability_consensus_results.json"
summary_csv_path = analysis_root / "stability_consensus_summary.csv"
source_csv_path = analysis_root / "stability_consensus_source_details.csv"

json_payload = None
if formatted_results.get("downloadResultsJson"):
    try:
        json_payload = json.loads(formatted_results["downloadResultsJson"])
    except Exception:
        json_payload = raw_results
else:
    json_payload = raw_results

json_path.write_text(json.dumps(json_payload, indent=2), encoding="utf-8")

summary_fields = [
    "cif_name",
    "composition",
    "space_group_number",
    "space_group_symbol",
    "materialsproject_id",
    "stable_count",
    "unstable_count",
    "total_sources",
    "consensus",
    "error",
]
with summary_csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=summary_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(summary_rows)

source_fields = [
    "cif_name",
    "composition",
    "source",
    "status",
    "stability",
    "stability_class",
    "raw_value",
    "threshold",
    "unit",
    "description",
    "message",
]
with source_csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=source_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(source_rows)

print(f"Saved JSON report: {json_path}")
print(f"Saved summary CSV: {summary_csv_path}")
print(f"Saved source-details CSV: {source_csv_path}")

In [ ]:
try:
    import pandas as pd

    summary_df = pd.DataFrame(summary_rows)
    source_df = pd.DataFrame(source_rows)

    print("Per-CIF summary (first 15 rows):")
    display(summary_df.head(15))

    print("Per-source stability distribution:")
    if not source_df.empty:
        display(source_df.groupby(["source", "stability_class"]).size().reset_index(name="count"))
    else:
        print("No source rows available.")

    print("Batch summary from feature:")
    display(pd.DataFrame([batch_summary]))

    print("Plot data from feature:")
    display(pd.DataFrame.from_dict(plot_data, orient="index").reset_index().rename(columns={"index": "source"}))
except Exception as exc:
    print("Pandas not available or display failed.")
    print(f"Reason: {exc}")
    print("Batch summary:")
    print(json.dumps(batch_summary, indent=2))